In [2]:

import pandas as pd
import random

random.seed(42)

locations = [
    "San Diego", "Irvine", "Los Angeles",
    "Anaheim", "Beverly Hills", "Santa Monica",
    "Pasadena", "Long Beach", "Costa Mesa", "Tustin"
]

browsing_templates = [
    "show me homes in {}",
    "houses for sale in {}",
    "explore properties in {}",
    "luxury homes in {}",
    "what homes are available in {}",
    "condos in {}",
    "townhouses in {}"
]

researching_templates = [
    "compare home prices in {}",
    "what are the property taxes in {}",
    "which neighborhoods in {} have low HOA fees",
    "average home prices in {}",
    "is {} a good place to buy a condo",
    "how competitive is the housing market in {}",
    "what are the closing costs for homes in {}"
]

high_intent_templates = [
    "schedule a home tour in {} this weekend",
    "I want to book a showing in {} tomorrow",
    "contact the seller of a home in {}",
    "can I tour a house in {} today",
    "find open houses in {} this Saturday",
    "I am ready to make an offer on a home in {}",
    "connect me with a listing agent in {}"
]

data = []

for location in locations:
    for template in browsing_templates:
        data.append((template.format(location), "browsing"))

    for template in researching_templates:
        data.append((template.format(location), "researching"))

    for template in high_intent_templates:
        data.append((template.format(location), "high_intent_inquiry"))

df = pd.DataFrame(data, columns=["query", "intent"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Total queries:", len(df))
print(df["intent"].value_counts())
display(df.head(10))


Total queries: 210
intent
researching            70
browsing               70
high_intent_inquiry    70
Name: count, dtype: int64


,query,intent
0,which neighborhoods in Irvine have low HOA fees,researching
1,what homes are available in Costa Mesa,browsing
2,show me homes in Beverly Hills,browsing
3,average home prices in Tustin,researching
4,find open houses in Los Angeles this Saturday,high_intent_inquiry
5,what are the property taxes in Long Beach,researching
6,luxury homes in Los Angeles,browsing
7,schedule a home tour in Costa Mesa this weekend,high_intent_inquiry
8,which neighborhoods in San Diego have low HOA ...,researching
9,compare home prices in Tustin,researching


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scripts.query_intent_classifier import QueryIntentClassifier

# Split dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    df["query"],
    df["intent"],
    test_size=0.2,
    random_state=42,
    stratify=df["intent"]
)

# Train classifier
classifier = QueryIntentClassifier()
classifier.train(X_train, y_train)

# Make predictions
predictions = [
    classifier.predict(query)[0]
    for query in X_test
]

# Evaluate model
accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Test Accuracy: 100.00%

Classification Report:
                     precision    recall  f1-score   support

           browsing       1.00      1.00      1.00        14
high_intent_inquiry       1.00      1.00      1.00        14
        researching       1.00      1.00      1.00        14

           accuracy                           1.00        42
          macro avg       1.00      1.00      1.00        42
       weighted avg       1.00      1.00      1.00        42


Confusion Matrix:
[[14  0  0]
 [ 0 14  0]
 [ 0  0 14]]


In [4]:
new_queries = [
    "just looking around at apartments near the beach",
    "trying to understand whether buying or renting is cheaper",
    "can someone arrange a viewing for tomorrow morning",
    "not sure what kind of property I want yet",
    "how do school districts affect home values",
    "I would like to speak with the agent about this listing",
    "any interesting properties around Orange County",
    "what are the pros and cons of buying a townhouse",
    "is this property still available for a showing today"
]

for query in new_queries:
    intent, confidence = classifier.predict(query)
    print(f"Query: {query}")
    print(f"Intent: {intent} | Confidence: {confidence:.2%}\n")

Query: just looking around at apartments near the beach
Intent: researching | Confidence: 46.28%

Query: trying to understand whether buying or renting is cheaper
Intent: researching | Confidence: 50.72%

Query: can someone arrange a viewing for tomorrow morning
Intent: high_intent_inquiry | Confidence: 51.14%

Query: not sure what kind of property I want yet
Intent: high_intent_inquiry | Confidence: 42.52%

Query: how do school districts affect home values
Intent: researching | Confidence: 47.67%

Query: I would like to speak with the agent about this listing
Intent: high_intent_inquiry | Confidence: 74.04%

Query: any interesting properties around Orange County
Intent: browsing | Confidence: 53.47%

Query: what are the pros and cons of buying a townhouse
Intent: researching | Confidence: 71.82%

Query: is this property still available for a showing today
Intent: high_intent_inquiry | Confidence: 43.75%



In [5]:
def predict_with_uncertainty(query, threshold=0.60):
    intent, confidence = classifier.predict(query)

    return {
        "query": query,
        "intent": intent,
        "confidence": round(confidence, 4),
        "uncertain": bool(confidence < threshold)
    }

for query in new_queries:
    print(predict_with_uncertainty(query))

{'query': 'just looking around at apartments near the beach', 'intent': 'researching', 'confidence': 0.4628, 'uncertain': True}
{'query': 'trying to understand whether buying or renting is cheaper', 'intent': 'researching', 'confidence': 0.5072, 'uncertain': True}
{'query': 'can someone arrange a viewing for tomorrow morning', 'intent': 'high_intent_inquiry', 'confidence': 0.5114, 'uncertain': True}
{'query': 'not sure what kind of property I want yet', 'intent': 'high_intent_inquiry', 'confidence': 0.4252, 'uncertain': True}
{'query': 'how do school districts affect home values', 'intent': 'researching', 'confidence': 0.4767, 'uncertain': True}
{'query': 'I would like to speak with the agent about this listing', 'intent': 'high_intent_inquiry', 'confidence': 0.7404, 'uncertain': False}
{'query': 'any interesting properties around Orange County', 'intent': 'browsing', 'confidence': 0.5347, 'uncertain': True}
{'query': 'what are the pros and cons of buying a townhouse', 'intent': 'resea

In [7]:

# Import sys so we can tell Python where to look for files.
import sys

# Import Path to work with folder locations.
from pathlib import Path

# Get the notebook's current working directory.
current_dir = Path.cwd()

# Check whether Python is currently inside the notebooks folder.
# If so, go up one level to reach the main project folder.
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Add the main project folder to Python's import search paths.
# This allows Python to find the scripts folder.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the Week 7 QueryIntentClassifier
# from query_intent_classifier.py inside scripts.
from scripts.query_intent_classifier import QueryIntentClassifier

# Initialize the Week 7 intent classifier.
query_intent_classifier = QueryIntentClassifier()

# Confirm that everything loaded successfully.
print("Week 7 QueryIntentClassifier initialized successfully!")


Week 7 QueryIntentClassifier initialized successfully!


In [8]:

# Import train_test_split to divide the dataset.
from sklearn.model_selection import train_test_split

# Split the queries into training and testing sets.
# 80% will be used for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    df["query"],
    df["intent"],
    test_size=0.20,
    random_state=42,
    stratify=df["intent"]
)

# Train the Week 7 classifier using the training queries.
query_intent_classifier.train(X_train, y_train)

# Confirm that training is complete.
print("Week 7 QueryIntentClassifier trained successfully!")


Week 7 QueryIntentClassifier trained successfully!


In [9]:

# Create a sample real estate search query.
query = "homes with open houses this weekend"

# Predict the user's likely search intent and confidence.
intent, confidence = query_intent_classifier.predict(query)

# Display the results.
print("Search Query:", query)
print("Predicted Intent:", intent)
print(f"Confidence: {confidence:.2%}")


Search Query: homes with open houses this weekend
Predicted Intent: high_intent_inquiry
Confidence: 72.21%


In [12]:

import pandas as pd

# Create additional real estate search queries with varied wording.
# These examples are designed to improve language diversity.

additional_data = {
    "query": [
        # ========================================
        # BROWSING — 20 examples
        # ========================================
        "just looking around at apartments near the beach",
        "not sure what kind of property I want yet",
        "what houses are out there right now",
        "curious about homes in Southern California",
        "show me something interesting near the ocean",
        "I am casually exploring different neighborhoods",
        "any nice places around Orange County",
        "just browsing condos with nice views",
        "what types of homes can I find near downtown",
        "I want to see some modern-looking houses",
        "looking at properties for inspiration",
        "show me a variety of homes in California",
        "any affordable apartments around here",
        "I wonder what luxury houses look like in Malibu",
        "exploring different housing options for the future",
        "not planning to buy yet, just checking listings",
        "can I see homes with large backyards",
        "show me popular neighborhoods with houses for sale",
        "I'm interested in seeing waterfront properties",
        "what listings are currently on the market",

        # ========================================
        # RESEARCHING — 20 examples
        # ========================================
        "trying to understand whether buying or renting is cheaper",
        "how do school districts affect home values",
        "what are the pros and cons of buying a townhouse",
        "which neighborhoods have the lowest property taxes",
        "how much are HOA fees for condos in Irvine",
        "is it cheaper to live in Anaheim or Irvine",
        "what should I know before buying a foreclosure",
        "compare mortgage costs for condos and single family homes",
        "how have home prices changed in San Diego",
        "which areas have the best long term resale value",
        "what are typical maintenance costs for older homes",
        "is buying a new construction home worth the premium",
        "how does flood risk affect home insurance",
        "what neighborhoods have good public transportation",
        "what is the average price per square foot in Pasadena",
        "compare property tax rates across Orange County",
        "how much should I budget for closing costs",
        "are townhomes generally cheaper to maintain than houses",
        "what are the advantages of a fixed rate mortgage",
        "which locations have experienced the fastest price growth",

        # ========================================
        # HIGH-INTENT INQUIRY — 20 examples
        # ========================================
        "can someone arrange a viewing for tomorrow morning",
        "I would like to speak with the agent about this listing",
        "is this property still available for a showing today",
        "I want to schedule a private tour this Friday",
        "can I visit the property before making an offer",
        "please contact me about the house on Main Street",
        "I am ready to submit an offer on this property",
        "can the listing agent call me this afternoon",
        "I want to see this condo in person tomorrow",
        "is there an open house I can attend this weekend",
        "please send me the disclosure documents for this house",
        "can you arrange a walkthrough with the seller",
        "I need to schedule a viewing before Monday",
        "how soon can I tour the property",
        "I would like to discuss the asking price with the agent",
        "can you connect me with someone handling this listing",
        "I am preapproved and want to tour this home",
        "please confirm whether I can visit the house Saturday",
        "I want to make an appointment to see this property",
        "can I get in touch with the seller about an offer"
    ],

    "intent": (
        ["browsing"] * 20 +
        ["researching"] * 20 +
        ["high_intent_inquiry"] * 20
    )
}

# Convert the manually written examples into a DataFrame.
additional_df = pd.DataFrame(additional_data)

# Combine the original dataset with the new examples.
df = pd.concat([df, additional_df], ignore_index=True)

# Remove any duplicate queries.
df = df.drop_duplicates(subset=["query"]).reset_index(drop=True)

# Display the updated dataset size.
print("Total queries:", len(df))

# Display the number of queries in each intent category.
print("\nIntent distribution:")
print(df["intent"].value_counts())

# Preview the new examples.
display(df.tail(10))


Total queries: 270

Intent distribution:
intent
researching            90
browsing               90
high_intent_inquiry    90
Name: count, dtype: int64


,query,intent
260,please send me the disclosure documents for th...,high_intent_inquiry
261,can you arrange a walkthrough with the seller,high_intent_inquiry
262,I need to schedule a viewing before Monday,high_intent_inquiry
263,how soon can I tour the property,high_intent_inquiry
264,I would like to discuss the asking price with ...,high_intent_inquiry
265,can you connect me with someone handling this ...,high_intent_inquiry
266,I am preapproved and want to tour this home,high_intent_inquiry
267,please confirm whether I can visit the house S...,high_intent_inquiry
268,I want to make an appointment to see this prop...,high_intent_inquiry
269,can I get in touch with the seller about an offer,high_intent_inquiry


In [13]:
# Save the expanded dataset for your Week 7 deliverable.
df.to_csv("week7_labeled_queries.csv", index=False)

# Confirm the dataset was saved.
print("Week 7 labeled queries saved successfully!")

Week 7 labeled queries saved successfully!


In [14]:

# Import the libraries needed to split and evaluate our dataset.
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import our Week 7 intent classifier.
from scripts.query_intent_classifier import QueryIntentClassifier

# Remove duplicate queries so the same query cannot appear
# in both the training and testing datasets.
df = df.drop_duplicates(subset=["query"]).reset_index(drop=True)

# Separate the original template-based queries from the
# 60 new conversational queries for a stronger evaluation.
# The original dataset had 210 generated queries.
# This assumes the new examples were appended afterward.
original_df = df.iloc[:210].copy()
new_df = df.iloc[210:].copy()

# Use the original queries for training.
X_train = original_df["query"]
y_train = original_df["intent"]

# Use the newer conversational queries for testing.
X_test = new_df["query"]
y_test = new_df["intent"]

# Initialize a new classifier.
classifier = QueryIntentClassifier()

# Train the model using the original queries.
classifier.train(X_train, y_train)

# Generate predictions on the newer queries.
predictions = [
    classifier.predict(query)[0]
    for query in X_test
]

# Calculate classification accuracy.
accuracy = accuracy_score(y_test, predictions)

# Display the evaluation results.
print(f"Test Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    predictions,
    labels=classifier.model.classes_
))


Test Accuracy: 75.00%

Classification Report:
                     precision    recall  f1-score   support

           browsing       0.80      0.40      0.53        20
high_intent_inquiry       0.71      1.00      0.83        20
        researching       0.77      0.85      0.81        20

           accuracy                           0.75        60
          macro avg       0.76      0.75      0.73        60
       weighted avg       0.76      0.75      0.73        60


Confusion Matrix:
[[ 8  7  5]
 [ 0 20  0]
 [ 2  1 17]]


In [15]:

# Create a list to store classification results.
results = []

# Predict the intent for each query in the test dataset.
for query, actual in zip(X_test, y_test):

    # Predict the intent and confidence score.
    intent, confidence = classifier.predict(query)

    # Store the prediction and actual label.
    results.append({
        "query": query,
        "actual_intent": actual,
        "predicted_intent": intent,
        "confidence": confidence,
        "correct": intent == actual
    })

# Convert results into a DataFrame.
results_df = pd.DataFrame(results)

# Display the first 20 predictions.
display(results_df.head(20))

# Identify predictions with confidence below 60%.
# This is an illustrative uncertainty threshold.
uncertain_queries = results_df[
    results_df["confidence"] < 0.60
]

# Display uncertain predictions.
print("Uncertain predictions:", len(uncertain_queries))
display(uncertain_queries)


,query,actual_intent,predicted_intent,confidence,correct
0,just looking around at apartments near the beach,browsing,researching,0.477006,False
1,not sure what kind of property I want yet,browsing,high_intent_inquiry,0.443901,False
2,what houses are out there right now,browsing,researching,0.391450,False
3,curious about homes in Southern California,browsing,browsing,0.804935,True
4,show me something interesting near the ocean,browsing,browsing,0.541504,True
5,I am casually exploring different neighborhoods,browsing,high_intent_inquiry,0.385017,False
6,any nice places around Orange County,browsing,high_intent_inquiry,0.358137,False
7,just browsing condos with nice views,browsing,browsing,0.517236,True
8,what types of homes can I find near downtown,browsing,high_intent_inquiry,0.508624,False
9,I want to see some modern-looking houses,browsing,high_intent_inquiry,0.679177,False


Uncertain predictions: 26


,query,actual_intent,predicted_intent,confidence,correct
0,just looking around at apartments near the beach,browsing,researching,0.477006,False
1,not sure what kind of property I want yet,browsing,high_intent_inquiry,0.443901,False
2,what houses are out there right now,browsing,researching,0.391450,False
4,show me something interesting near the ocean,browsing,browsing,0.541504,True
5,I am casually exploring different neighborhoods,browsing,high_intent_inquiry,0.385017,False
6,any nice places around Orange County,browsing,high_intent_inquiry,0.358137,False
7,just browsing condos with nice views,browsing,browsing,0.517236,True
8,what types of homes can I find near downtown,browsing,high_intent_inquiry,0.508624,False
12,any affordable apartments around here,browsing,high_intent_inquiry,0.358137,False
14,exploring different housing options for the fu...,browsing,researching,0.595439,False


In [16]:

# Import joblib to save the trained classifier.
import joblib

# Create a folder to store trained models.
models_folder = project_root / "models"
models_folder.mkdir(parents=True, exist_ok=True)

# Define the path for the Week 7 classifier.
model_path = models_folder / "week7_intent_classifier.joblib"

# Save the trained classifier.
joblib.dump(classifier, model_path)

# Confirm the model was saved.
print("Model saved to:", model_path)


Model saved to: c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\models\week7_intent_classifier.joblib


In [17]:

# Locate the existing Week 4 query parser.
parser_file = project_root / "scripts" / "query_parser.py"

# Read the query parser source code.
parser_code = parser_file.read_text(encoding="utf-8")

# Display the source code so we can identify its API.
print(parser_code)


"""Rule-based natural-language real estate query parser and safe SQL compiler."""
import re
from typing import Any, Mapping


class QueryParser:
    """Parse listing queries without executing SQL or trusting user-controlled identifiers."""

    AMENITIES = {
        'pool': (r'pool|swimming pool',),
        'garage': (r'garage|parking garage',),
        'parking': (r'parking|parking space',),
        'yard': (r'yard|backyard|garden',),
        'fireplace': (r'fireplace',),
        'balcony': (r'balcony',),
        'patio': (r'patio',),
        'solar': (r'solar(?: panels)?',),
        'air conditioning': (r'air conditioning|central air|a/c',),
        'view': (r'(?:ocean|mountain|city) view|view',),
        'laundry': (r'laundry|washer(?:/dryer)?',),
    }

    # Identifiers here are developer-controlled, never composed from user input.
    DEFAULT_COLUMNS = {
        'price': 'L_SystemPrice',
        'bedrooms': 'L_Keyword2',
        'bathrooms': 'L_Keyword3',  # VERIFY against the ac

In [18]:
# Select queries where the predicted intent differs
# from the actual intent label.
errors = results_df[
    results_df["correct"] == False
].copy()

# Display the misclassified queries and confidence scores.
display(
    errors[
        ["query", "actual_intent", "predicted_intent", "confidence"]
    ]
)

# Count the number of classification errors.
print("Total misclassified queries:", len(errors))

,query,actual_intent,predicted_intent,confidence
0,just looking around at apartments near the beach,browsing,researching,0.477006
1,not sure what kind of property I want yet,browsing,high_intent_inquiry,0.443901
2,what houses are out there right now,browsing,researching,0.391450
5,I am casually exploring different neighborhoods,browsing,high_intent_inquiry,0.385017
6,any nice places around Orange County,browsing,high_intent_inquiry,0.358137
8,what types of homes can I find near downtown,browsing,high_intent_inquiry,0.508624
9,I want to see some modern-looking houses,browsing,high_intent_inquiry,0.679177
12,any affordable apartments around here,browsing,high_intent_inquiry,0.358137
14,exploring different housing options for the fu...,browsing,researching,0.595439
15,"not planning to buy yet, just checking listings",browsing,researching,0.566496


Total misclassified queries: 15


In [20]:
# Import the libraries needed for model tuning.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Create a pipeline combining TF-IDF and Logistic Regression.
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Define the parameters to test.
param_grid = {
    "tfidf__max_features": [500, 1000, 2000],
    "tfidf__ngram_range": [(1, 2), (1, 3)],
    "tfidf__sublinear_tf": [True, False],
    "model__C": [0.1, 1, 10]
}

# Perform 5-fold cross-validation on training data.
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

# Train and tune the classifier.
grid_search.fit(X_train, y_train)

# Select the best model.
best_model = grid_search.best_estimator_

# Predict on the separate test set.
predictions = best_model.predict(X_test)

# Evaluate test accuracy.
accuracy = accuracy_score(y_test, predictions)

print("Best Parameters:", grid_search.best_params_)
print(f"Cross-validation Accuracy: {grid_search.best_score_:.2%}")
print(f"Test Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Best Parameters: {'model__C': 0.1, 'tfidf__max_features': 500, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Cross-validation Accuracy: 100.00%
Test Accuracy: 76.67%

Classification Report:
                     precision    recall  f1-score   support

           browsing       0.82      0.45      0.58        20
high_intent_inquiry       0.71      1.00      0.83        20
        researching       0.81      0.85      0.83        20

           accuracy                           0.77        60
          macro avg       0.78      0.77      0.75        60
       weighted avg       0.78      0.77      0.75        60



In [21]:

# Import pandas to organize our new training examples.
import pandas as pd

# Create additional browsing examples with different wording.
# These are suggested labels that should be manually reviewed.
new_browsing_queries = [
    "I'm casually checking out different properties",
    "just exploring what homes are available",
    "not planning on buying anytime soon",
    "I haven't decided what type of house I want",
    "I'm only looking for ideas right now",
    "show me some random properties in California",
    "curious to see what apartments are available",
    "I'm browsing homes without a specific budget",
    "just want to look at some nice houses",
    "I wonder what properties are on the market",
    "looking around without any specific requirements",
    "I am exploring my housing options",
    "no rush, just checking out available houses",
    "not ready to buy, only browsing",
    "show me a few interesting listings",
    "I want to get a feel for the housing market",
    "just checking what kinds of houses exist",
    "I have no particular property in mind",
    "casually looking through real estate listings",
    "show me different types of homes near the coast",
    "I might buy someday but I'm just looking now",
    "I'm exploring places to live in the future",
    "show me whatever houses are available",
    "I don't have a preferred neighborhood yet",
    "I'm browsing properties for inspiration",
    "what interesting homes are currently listed",
    "I'm curious about real estate in this area",
    "just scrolling through available homes",
    "I want to see what's out there",
    "not sure where I want to live yet"
]

# Convert the new examples into a labeled DataFrame.
new_browsing_df = pd.DataFrame({
    "query": new_browsing_queries,
    "intent": ["browsing"] * len(new_browsing_queries)
})

# Add these examples to the training data only.
# Do not add them to the existing test set.
X_train_updated = pd.concat([
    X_train.reset_index(drop=True),
    new_browsing_df["query"]
], ignore_index=True)

y_train_updated = pd.concat([
    y_train.reset_index(drop=True),
    new_browsing_df["intent"]
], ignore_index=True)

# Confirm the updated training dataset.
print("Training queries:", len(X_train_updated))
print("\nIntent distribution:")
print(y_train_updated.value_counts())


Training queries: 240

Intent distribution:
intent
browsing               100
researching             70
high_intent_inquiry     70
Name: count, dtype: int64


In [22]:

# Import the machine learning components.
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Build the classifier using the selected parameters.
updated_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=500,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("model", LogisticRegression(
        C=0.1,
        max_iter=1000,
        random_state=42
    ))
])

# Train the updated model on the expanded training dataset.
updated_model.fit(X_train_updated, y_train_updated)

# Predict intent on the existing diagnostic test queries.
updated_predictions = updated_model.predict(X_test)

# Calculate accuracy.
updated_accuracy = accuracy_score(y_test, updated_predictions)

# Display the results.
print(f"Updated Test Accuracy: {updated_accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, updated_predictions))


Updated Test Accuracy: 61.67%

Classification Report:
                     precision    recall  f1-score   support

           browsing       0.47      1.00      0.63        20
high_intent_inquiry       1.00      0.60      0.75        20
        researching       1.00      0.25      0.40        20

           accuracy                           0.62        60
          macro avg       0.82      0.62      0.59        60
       weighted avg       0.82      0.62      0.59        60



In [23]:

# Import the required machine learning libraries.
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Create a new pipeline with balanced class weights.
# This reduces the influence of unequal class frequencies.
balanced_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=500,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("model", LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Train the model using the expanded training dataset.
balanced_model.fit(X_train_updated, y_train_updated)

# Generate predictions on the existing diagnostic set.
balanced_predictions = balanced_model.predict(X_test)

# Calculate accuracy.
balanced_accuracy = accuracy_score(y_test, balanced_predictions)

# Display the results.
print(f"Diagnostic Accuracy: {balanced_accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, balanced_predictions))


Diagnostic Accuracy: 90.00%

Classification Report:
                     precision    recall  f1-score   support

           browsing       0.77      1.00      0.87        20
high_intent_inquiry       1.00      1.00      1.00        20
        researching       1.00      0.70      0.82        20

           accuracy                           0.90        60
          macro avg       0.92      0.90      0.90        60
       weighted avg       0.92      0.90      0.90        60



In [24]:

# Import joblib to save our trained classifier.
import joblib

# Create the models folder if it does not already exist.
models_folder = project_root / "models"
models_folder.mkdir(parents=True, exist_ok=True)

# Define the filename for the Week 7 balanced classifier.
model_path = models_folder / "week7_intent_classifier.joblib"

# Save the trained TF-IDF and Logistic Regression pipeline.
joblib.dump(balanced_model, model_path)

# Confirm that the model was saved successfully.
print("Week 7 classifier saved successfully!")
print("Model location:", model_path)


Week 7 classifier saved successfully!
Model location: c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\models\week7_intent_classifier.joblib


In [25]:

# Import pandas to organize prediction results.
import pandas as pd

# Define sample real estate search queries.
sample_queries = [
    "I'm just casually looking at homes near the beach",
    "what are the average property taxes in Irvine",
    "can I schedule a tour of this house tomorrow",
    "compare condos and townhouses in San Diego",
    "I want to contact the listing agent today"
]

# Predict intent for each search query.
predictions = balanced_model.predict(sample_queries)

# Calculate the predicted probabilities for each category.
probabilities = balanced_model.predict_proba(sample_queries)

# Create a table displaying the classification results.
results = pd.DataFrame({
    "query": sample_queries,
    "predicted_intent": predictions,
    "confidence": probabilities.max(axis=1)
})

# Flag queries with confidence below 60%.
results["uncertain"] = results["confidence"] < 0.60

# Display the results.
display(results)


,query,predicted_intent,confidence,uncertain
0,I'm just casually looking at homes near the beach,browsing,0.670348,False
1,what are the average property taxes in Irvine,researching,0.853052,False
2,can I schedule a tour of this house tomorrow,high_intent_inquiry,0.858922,False
3,compare condos and townhouses in San Diego,browsing,0.778624,False
4,I want to contact the listing agent today,high_intent_inquiry,0.814297,False


In [26]:

# Import inspect to examine the existing Week 4 code.
import inspect

# Import the Week 4 query parser module.
from scripts import query_parser

# Find classes defined inside the query parser.
for name, obj in inspect.getmembers(query_parser, inspect.isclass):

    # Only include classes defined in this module.
    if obj.__module__ == query_parser.__name__:

        # Display the class name.
        print(f"\nClass: {name}")

        # Display its public methods and arguments.
        for method_name, method in inspect.getmembers(
            obj, inspect.isfunction
        ):
            if not method_name.startswith("_"):
                print(
                    method_name,
                    inspect.signature(method)
                )



Class: QueryParser
parse (self, query: str) -> dict[str, typing.Any]
to_sql (self, filters: Mapping[str, Any], validator=None) -> tuple[str, tuple[typing.Any, ...]]


In [27]:

# Import the Week 4 QueryParser to extract structured
# information from real estate search queries.
from scripts.query_parser import QueryParser

# Import pandas to display the combined results.
import pandas as pd

# Initialize the Week 4 query parser.
query_parser = QueryParser()

# The balanced_model from Week 7 is already trained.
# We will reuse it to predict search intent.

# Define a function combining Week 4 and Week 7.
def analyze_search_query(query):

    # Week 4: Extract structured information from the query.
    parsed_filters = query_parser.parse(query)

    # Week 7: Predict the user's probable search intent.
    predicted_intent = balanced_model.predict([query])[0]

    # Week 7: Calculate the probability of each intent category.
    probabilities = balanced_model.predict_proba([query])[0]

    # Select the highest predicted probability.
    confidence = float(probabilities.max())

    # Combine structured filters and predicted intent.
    return {
        "query": query,
        "parsed_filters": parsed_filters,
        "predicted_intent": predicted_intent,
        "confidence": round(confidence, 4),
        "uncertain": bool(confidence < 0.60)
    }

# Test the integrated system on a sample query.
test_query = "3 bedroom homes in Irvine under $900k with open houses this weekend"

# Analyze the query using both Week 4 and Week 7.
result = analyze_search_query(test_query)

# Display the combined output.
for key, value in result.items():
    print(f"{key}: {value}")


query: 3 bedroom homes in Irvine under $900k with open houses this weekend
parsed_filters: {'price_max': 900000, 'bedrooms': 3, 'city': 'Irvine'}
predicted_intent: high_intent_inquiry
confidence: 0.6177
uncertain: False


In [28]:

# Define several queries representing different search intents.
test_queries = [
    "show me houses in San Diego",
    "compare condo prices in Irvine",
    "schedule a home tour tomorrow",
    "3 bedroom homes in Irvine under $900k",
    "what are property taxes like in Anaheim"
]

# Analyze each query using both modules.
combined_results = [
    analyze_search_query(query)
    for query in test_queries
]

# Convert the results into a DataFrame.
results_df = pd.DataFrame(combined_results)

# Display the combined output.
display(results_df)


,query,parsed_filters,predicted_intent,confidence,uncertain
0,show me houses in San Diego,{'city': 'San Diego'},browsing,0.6441,False
1,compare condo prices in Irvine,{'city': 'Irvine'},researching,0.7510,False
2,schedule a home tour tomorrow,{},high_intent_inquiry,0.8091,False
3,3 bedroom homes in Irvine under $900k,"{'price_max': 900000, 'bedrooms': 3, 'city': '...",browsing,0.7263,False
4,what are property taxes like in Anaheim,{'city': 'Anaheim'},researching,0.6774,False


In [29]:

# Confirm that integration returns the required fields.
required_fields = [
    "query",
    "parsed_filters",
    "predicted_intent",
    "confidence",
    "uncertain"
]

# Test the combined query analysis function.
result = analyze_search_query(
    "homes in Irvine under $800k"
)

# Check whether all required fields exist.
assert all(field in result for field in required_fields)

# Confirm the parsed output is a dictionary.
assert isinstance(result["parsed_filters"], dict)

# Confirm the predicted intent is one of the allowed categories.
assert result["predicted_intent"] in [
    "browsing",
    "researching",
    "high_intent_inquiry"
]

# Confirm the confidence score is between 0 and 1.
assert 0 <= result["confidence"] <= 1

# Display confirmation.
print("Week 4 + Week 7 integration test passed!")


Week 4 + Week 7 integration test passed!


In [31]:

# Import pandas to organize our final test dataset.
import pandas as pd

# Create new real estate search queries with varied wording.
# These examples are intended for independent manual review.
final_queries = [
    # Browsing candidates
    "I'd love to see what kinds of lofts are out there",
    "I'm window shopping for a place with mountain views",
    "show me some unusual homes across the state",
    "wondering what residential properties are listed nearby",
    "just having a look at available vacation cottages",
    "I'm exploring housing styles without any plans to move",
    "what sort of apartments are on the market these days",
    "show me properties with interesting architecture",
    "I'm curious what homes are like in smaller towns",
    "looking at houses for fun while planning my future",

    # Researching candidates
    "how does the age of a building affect its resale price",
    "which type of property has lower insurance expenses",
    "does proximity to a train station increase property value",
    "how do condo association rules differ by neighborhood",
    "what factors make a home more expensive to maintain",
    "is a duplex generally a better investment than a condo",
    "how does the local job market influence housing demand",
    "what are the financial tradeoffs of buying an older house",
    "which housing styles tend to appreciate more slowly",
    "how do renovation costs compare between different homes",

    # High-intent inquiry candidates
    "could an agent confirm a tour appointment for Thursday",
    "I'd like to submit a purchase proposal for this address",
    "please arrange a private inspection of the listed house",
    "who should I call to discuss purchasing this unit",
    "I have financing arranged and would like to visit today",
    "can you help me book an appointment with the property owner",
    "I'd like to request the seller's disclosure before offering",
    "is someone available to show me this address on Friday",
    "please put me in contact with the agent representing this home",
    "I'd like to start the purchase process for this listing"
]

# Display the number of candidate queries.
print("Total candidate test queries:", len(final_queries))


Total candidate test queries: 30


In [32]:

# Import interactive Jupyter widgets.
import ipywidgets as widgets
from IPython.display import display, clear_output

# Store manually selected labels.
manual_labels = {}

# Track the current query being reviewed.
current_index = 0

# Create a label displaying the current query.
query_display = widgets.HTML()

# Create a dropdown for selecting intent.
intent_dropdown = widgets.Dropdown(
    options=[
        ("Select intent", ""),
        ("Browsing", "browsing"),
        ("Researching", "researching"),
        ("High-Intent Inquiry", "high_intent_inquiry")
    ],
    value="",
    description="Intent:"
)

# Create navigation buttons.
previous_button = widgets.Button(description="Previous")
save_button = widgets.Button(
    description="Save & Next",
    button_style="success"
)

# Create a status message.
status_display = widgets.HTML()

# Update the widget to show the current query.
def show_query():
    query_display.value = (
        f"<h4>Query {current_index + 1} of {len(final_queries)}</h4>"
        f"<p>{final_queries[current_index]}</p>"
    )

    intent_dropdown.value = manual_labels.get(current_index, "")

    status_display.value = (
        f"Reviewed: {len(manual_labels)} / {len(final_queries)}"
    )

# Save the selected intent and move to the next query.
def save_and_next(button):
    global current_index

    if not intent_dropdown.value:
        status_display.value = "Please select an intent first."
        return

    manual_labels[current_index] = intent_dropdown.value

    if current_index < len(final_queries) - 1:
        current_index += 1

    show_query()

# Move to the previous query.
def go_previous(button):
    global current_index

    if current_index > 0:
        current_index -= 1

    show_query()

# Connect buttons to their functions.
save_button.on_click(save_and_next)
previous_button.on_click(go_previous)

# Display the interactive labeling tool.
show_query()
display(
    widgets.VBox([
        query_display,
        intent_dropdown,
        widgets.HBox([previous_button, save_button]),
        status_display
    ])
)


In [33]:

# Check that every test query has been manually labeled.
if len(manual_labels) != len(final_queries):
    print(
        f"Please finish labeling. "
        f"{len(manual_labels)}/{len(final_queries)} completed."
    )

else:
    # Create a DataFrame containing the queries and labels.
    final_test_df = pd.DataFrame({
        "query": final_queries,
        "intent": [
            manual_labels[i]
            for i in range(len(final_queries))
        ]
    })

    # Display the number of examples in each category.
    print("Final test dataset created!")
    print(final_test_df["intent"].value_counts())

    # Display the dataset.
    display(final_test_df)


Final test dataset created!
intent
browsing               10
researching            10
high_intent_inquiry    10
Name: count, dtype: int64


,query,intent
0,I'd love to see what kinds of lofts are out there,browsing
1,I'm window shopping for a place with mountain ...,browsing
2,show me some unusual homes across the state,browsing
3,wondering what residential properties are list...,browsing
4,just having a look at available vacation cottages,browsing
5,I'm exploring housing styles without any plans...,browsing
6,what sort of apartments are on the market thes...,browsing
7,show me properties with interesting architecture,browsing
8,I'm curious what homes are like in smaller towns,browsing
9,looking at houses for fun while planning my fu...,browsing


In [34]:

# Import the metrics needed to evaluate the final classifier.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Use the final, independently labeled test queries.
X_final = final_test_df["query"]
y_final = final_test_df["intent"]

# Predict the intent using the trained balanced model.
final_predictions = balanced_model.predict(X_final)

# Calculate accuracy on the independent test set.
final_accuracy = accuracy_score(y_final, final_predictions)

# Display the final evaluation results.
print(f"Final Test Accuracy: {final_accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(
    y_final,
    final_predictions,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_final,
    final_predictions,
    labels=balanced_model.classes_
))

# Check whether the required 80% accuracy target was met.
print("\n80% Accuracy Target Met:", final_accuracy >= 0.80)


Final Test Accuracy: 80.00%

Classification Report:
                     precision    recall  f1-score   support

           browsing       0.67      1.00      0.80        10
high_intent_inquiry       0.90      0.90      0.90        10
        researching       1.00      0.50      0.67        10

           accuracy                           0.80        30
          macro avg       0.86      0.80      0.79        30
       weighted avg       0.86      0.80      0.79        30


Confusion Matrix:
[[10  0  0]
 [ 1  9  0]
 [ 4  1  5]]

80% Accuracy Target Met: True


In [35]:

# Import libraries for saving data and trained models.
import pandas as pd
import joblib

# Define the folders for processed data and trained models.
output_folder = project_root / "data" / "processed"
model_folder = project_root / "models"

# Create the folders if they do not exist.
output_folder.mkdir(parents=True, exist_ok=True)
model_folder.mkdir(parents=True, exist_ok=True)

# Save the labeled training dataset.
df.to_csv(
    output_folder / "week7_labeled_queries.csv",
    index=False
)

# Save the trained balanced classifier.
joblib.dump(
    balanced_model,
    model_folder / "week7_intent_classifier.joblib"
)

# Calculate confidence scores for final test predictions.
final_probabilities = balanced_model.predict_proba(
    final_test_df["query"]
)

# Create the final evaluation results.
final_results = final_test_df.copy()

final_results["predicted_intent"] = final_predictions
final_results["confidence"] = final_probabilities.max(axis=1)

# Save final evaluation results.
final_results.to_csv(
    output_folder / "week7_final_evaluation.csv",
    index=False
)

# Confirm successful export.
print("Week 7 files saved successfully!")
print("Final Test Accuracy:", f"{final_accuracy:.2%}")


Week 7 files saved successfully!
Final Test Accuracy: 80.00%
